# ECG Multimodal Learning: ECG Signal + Clinical Text (v3)
**Architecture**: an ECG encoder (MERL's 1-D ResNet18 by default, or the ECG-FM CNN + 12-layer Transformer) paired with a clinical text encoder (MedCPT by default, or Bio_ClinicalBERT) via CLIP-style contrastive learning. Defaults = best configuration on MERL's benchmark (`results/MERL_COMPARISON.md`).

**Dataset**: PTB-XL (21,799 12-lead ECGs, 500 Hz) + cardiologist free-text reports (translated to English) or SCP statement descriptions.

**v2 changes (validated overnight, see `experiments/RESULTS.md`)**
- **Label fix**: superclass labels now come from `scp_statements.csv → diagnostic_class` (v1 looked for a non-existent `diagnostic_superclass` column and silently used a wrong hand-written map). Counts now match the official PTB-XL numbers.
- **Evaluation fix**: v1's `classification_report(..., target_names=LABEL_ORDER)` assigned names alphabetically, so every row was mislabelled. v2 reports the standard PTB-XL metric (**macro AUROC**, multi-label) plus single-label accuracy with correct names.
- **Deterministic evaluation**: each test ECG is embedded as the average of its two 5-s windows (v1 used a random crop, so results changed run to run).
- **Text**: option to train on the real cardiologist reports (`TEXT_SOURCE='report'`, translated from German/Swedish) instead of text generated from the SCP labels. Also fixes the sex mapping (PTB-XL: 0 = male, 1 = female; v1 had it flipped).
- **Multi-positive contrastive loss**: ECGs in a batch with an identical report are no longer treated as negatives of each other (correct objective; the ablation showed no measurable gain with report text, so it is not claimed as an improvement).
- **Optimisation**: higher LR for the from-scratch ECG encoder, learnable temperature with its own LR (in v1 it never moved), bf16 mixed precision, optional gradient checkpointing, signal cache (no `wfdb` reads during training).
- **Baselines**: random-initialised encoder linear probe, and linear probe with 1% / 10% / 100% of the labels.

**v3 additions (from MERL, Liu et al. ICML 2024; see `results/MERL_COMPARISON.md`)**
- `TEXT_MODEL_NAME` can be MERL's text encoder `ncbi/MedCPT-Query-Encoder`; `FREEZE_TEXT` keeps it frozen (as MERL's code does).
- `USE_UMA`: MERL's uni-modal alignment loss (two dropout views of the pooled ECG feature, contrasted with each other).
- `ECG_ARCH='resnet18'`: MERL's 1-D ResNet18 backbone instead of the ECG-FM Transformer.
- Zero-shot also with MERL's released CKEPE prompts (`CKEPE_JSON`).

## 0. Setup (Colab)

In [ ]:
# Run this cell only in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q wfdb transformers accelerate einops scikit-learn umap-learn
    from google.colab import drive
    drive.mount('/content/drive')

## 1. Configuration

In [ ]:
!cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip" .

In [ ]:
!unzip -q ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip -d ptbxl

In [ ]:
import os, sys, json, math, time, random, ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

IN_COLAB = 'google.colab' in sys.modules

# ── Paths ──────────────────────────────────────────────────────────────────
if IN_COLAB:
    PTB_DIR   = '/content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
    SAVE_DIR  = '/content/drive/MyDrive/ecg_multimodal_checkpoints'
    CACHE_DIR = '/content/cache'                                   # local disk (fast)
    REPORTS_EN_CSV = '/content/drive/MyDrive/ptbxl_reports_en.csv' # upload data/ptbxl_reports_en.csv here
    CKEPE_JSON     = '/content/drive/MyDrive/CKEPE_prompt.json'     # merl_ref/zeroshot/CKEPE_prompt.json (MERL, MIT licence)
else:
    PTB_DIR   = r'C:\Users\eq26\Desktop\ECG LM\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
    SAVE_DIR  = r'C:\Users\eq26\Desktop\ECG LM\ecg_multimodal_checkpoints'
    CACHE_DIR = r'C:\Users\eq26\Desktop\ECG LM\cache'
    REPORTS_EN_CSV = r'C:\Users\eq26\Desktop\ECG LM\ptbxl_reports_en.csv'
    CKEPE_JSON     = r'C:\Users\eq26\Desktop\ECG LM\CKEPE_prompt.json'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ── ECG ────────────────────────────────────────────────────────────────────
TARGET_FS   = 500
SEGMENT_LEN = 2500          # 5 s windows; each record is 10 s = 2 windows
N_LEADS     = 12

# ── Model ──────────────────────────────────────────────────────────────────
PROJ_DIM        = 512
TEXT_MODEL_NAME = 'ncbi/MedCPT-Query-Encoder'   # 'ncbi/MedCPT-Query-Encoder' (MERL's choice, best here) | 'emilyalsentzer/Bio_ClinicalBERT'
FREEZE_TEXT     = False        # True = frozen text encoder, only the projection head learns (MERL's code)
TEXT_MAX_LEN    = 128
ECG_ARCH        = 'resnet18'           # 'fm' = ECG-FM CNN+Transformer (86M) | 'resnet18' = MERL's 1-D ResNet18 (4M)
USE_UMA         = False        # MERL uni-modal alignment loss on the ECG side

# ── Text supervision ───────────────────────────────────────────────────────
# 'report' : cardiologist free-text report, translated to English (experiments/translate_reports.py)
# 'scp'    : sentence built from the SCP statement descriptions (diagnostic + form + rhythm)
#            NOTE: 'scp' text is generated from the same labels used for evaluation, so its
#            zero-shot numbers are optimistic (label leakage). 'report' is the honest setting.
TEXT_SOURCE    = 'report'
MULTI_POSITIVE = True   # identical texts in a batch are positives, not negatives

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE    = 64
NUM_EPOCHS    = 5
WARMUP_EPOCHS = 1
LR_ECG        = 0.0002      # ECG encoder is trained from scratch -> needs a larger LR
LR_TEXT       = 2e-05      # pretrained BERT -> small LR
LR_PROJ       = 0.0002
LR_TEMP       = 0.001      # learnable temperature (v1 used 2e-5, so it never moved)
MIN_LR_FRAC   = 0.02
INIT_TEMP     = 0.07
WEIGHT_DECAY  = 1e-4
MAX_GRAD_NORM = 1.0
USE_BF16      = True       # bf16 autocast (Ampere or newer GPUs)
GRAD_CKPT     = False      # True on GPUs with < 12 GB (e.g. 6 GB laptop GPU)
SEED          = 0

SUPERS = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f'Device: {DEVICE}', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. PTB-XL Metadata, Labels & Text

In [ ]:
ptb_df = pd.read_csv(os.path.join(PTB_DIR, 'ptbxl_database.csv'), index_col='ecg_id')
ptb_df['scp_codes'] = ptb_df['scp_codes'].apply(ast.literal_eval)
scp_df = pd.read_csv(os.path.join(PTB_DIR, 'scp_statements.csv'), index_col=0)
print(f'Total records: {len(ptb_df)} | SCP statements: {len(scp_df)}')

In [ ]:
# ── Superclass labels (standard PTB-XL "superdiagnostic" task, multi-label) ──
# Only diagnostic statements count; the class comes from scp_statements.csv -> diagnostic_class.
diag = scp_df[scp_df['diagnostic'] == 1]
CODE_TO_SUPER = diag['diagnostic_class'].to_dict()

Y = np.zeros((len(ptb_df), len(SUPERS)), dtype=np.float32)
for i, codes in enumerate(ptb_df['scp_codes']):
    for c in codes:
        if c in CODE_TO_SUPER:
            Y[i, SUPERS.index(CODE_TO_SUPER[c])] = 1

print('Records per superclass (a record can have several):')
print(dict(zip(SUPERS, Y.sum(0).astype(int))))   # official: NORM 9514, MI 5469, STTC 5235, CD 4898, HYP 2649
print('Records with no superclass label:', int((Y.sum(1) == 0).sum()))

In [ ]:
# ── Text for each ECG ─────────────────────────────────────────────────────
DESC    = scp_df['description'].to_dict()
IS_DIAG = scp_df['diagnostic'].fillna(0).to_dict()
IS_FORM = scp_df['form'].fillna(0).to_dict()
IS_RHY  = scp_df['rhythm'].fillna(0).to_dict()


def scp_text(codes):
    """All SCP statements: diagnostic (by likelihood), then form, then rhythm.
    v1 dropped every statement with likelihood < 50, which removed ALL rhythm/form codes
    (PTB-XL stores them with likelihood 0), e.g. 'sinus rhythm' or 'atrial fibrillation'."""
    d  = sorted([c for c in codes if IS_DIAG.get(c)], key=lambda c: -codes[c])
    fm = [c for c in codes if IS_FORM.get(c) and not IS_DIAG.get(c)]
    rh = [c for c in codes if IS_RHY.get(c)]
    phrases = [DESC[c] for c in d + fm + rh if c in DESC]
    return '. '.join(dict.fromkeys(phrases)) + '.' if phrases else 'electrocardiogram.'


if TEXT_SOURCE == 'report':
    rep = pd.read_csv(REPORTS_EN_CSV, index_col='ecg_id').reindex(ptb_df.index)
    texts = [t if isinstance(t, str) and t.strip() else 'electrocardiogram.' for t in rep['report_en']]
else:
    texts = [scp_text(c) for c in ptb_df['scp_codes']]

ptb_df['text'] = texts
TEXT_IDS = pd.factorize(pd.Series(texts))[0]      # same id <=> identical text (used by the multi-positive loss)
print(f'TEXT_SOURCE={TEXT_SOURCE} | unique texts: {TEXT_IDS.max() + 1}')
for i in [0, 10, 100, 1000]:
    print(f'  ECG {ptb_df.index[i]}: {texts[i]}')

## 3. Signal Cache (preprocess once)

In [ ]:
import wfdb

CACHE_FILE = os.path.join(CACHE_DIR, 'ptbxl_500hz_zscore_f16.npy')


def preprocess_record(sig):
    """[5000, 12] -> [12, 5000] float32, NaN-safe, z-scored per lead over the full 10 s."""
    x = np.asarray(sig, dtype=np.float32).T
    if not np.isfinite(x).all():
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-8)
    return np.clip(x, -50, 50)


if not os.path.exists(CACHE_FILE):
    arr = np.lib.format.open_memmap(CACHE_FILE + '.tmp.npy', mode='w+', dtype=np.float16,
                                    shape=(len(ptb_df), N_LEADS, 5000))
    t0 = time.time()
    for i, fn in enumerate(ptb_df['filename_hr']):
        sig, meta = wfdb.rdsamp(os.path.join(PTB_DIR, fn))
        assert meta['fs'] == TARGET_FS
        arr[i] = preprocess_record(sig).astype(np.float16)
        if i % 5000 == 0:
            print(f'  cached {i}/{len(ptb_df)} ({time.time() - t0:.0f}s)')
    arr.flush(); del arr
    os.replace(CACHE_FILE + '.tmp.npy', CACHE_FILE)

SIG = np.load(CACHE_FILE)          # ~2.6 GB in RAM, float16
print('Signal cache:', SIG.shape, SIG.dtype)

## 4. ECG Encoder (ResNet18 or ECG-FM Transformer)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, channels=256, kernel_size=2, stride=2):
        super().__init__()
        self.conv = nn.Conv1d(channels, channels, kernel_size, stride)
        self.norm = nn.GroupNorm(1, channels)
        self.act  = nn.GELU()

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class ECGFeatureExtractor(nn.Module):
    def __init__(self, in_channels=12, hidden=256):
        super().__init__()
        self.proj = nn.Conv1d(in_channels, hidden, 2, 2)
        self.norm = nn.GroupNorm(1, hidden)
        self.act  = nn.GELU()
        self.blocks = nn.Sequential(ConvBlock(hidden), ConvBlock(hidden), ConvBlock(hidden))

    def forward(self, x):          # x: [B, 12, 2500]
        return self.blocks(self.act(self.norm(self.proj(x))))   # [B, 256, 156]


class RelativePositionalConv(nn.Module):
    def __init__(self, embed_dim=768, pos_dim=128, kernel=128, groups=16):
        super().__init__()
        self.in_  = nn.Linear(embed_dim, pos_dim)
        self.conv = nn.Conv1d(pos_dim, pos_dim, kernel, padding=kernel // 2, groups=groups)
        self.act  = nn.GELU()
        self.out_ = nn.Linear(pos_dim, embed_dim)

    def forward(self, x):          # x: [B, T, D]
        p = self.in_(x).transpose(1, 2)
        p = self.conv(p)[..., :x.size(1)]
        p = self.act(p).transpose(1, 2)
        return x + self.out_(p)


class TransformerBlock(nn.Module):
    def __init__(self, dim=768, heads=12, ff_dim=3072, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.ff    = nn.Sequential(nn.Linear(dim, ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim, dim))
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop1(h)
        return x + self.drop2(self.ff(self.norm2(x)))


class ECGEncoder(nn.Module):
    """CNN feature extractor + 12-layer Transformer; returns the mean-pooled [B, 768] embedding."""
    def __init__(self, in_ch=12, cnn_dim=256, embed_dim=768, heads=12, ff_dim=3072, n_layers=12, dropout=0.1):
        super().__init__()
        self.cnn    = ECGFeatureExtractor(in_ch, cnn_dim)
        self.proj   = nn.Linear(cnn_dim, embed_dim)
        self.pos    = RelativePositionalConv(embed_dim)
        self.layers = nn.ModuleList([TransformerBlock(embed_dim, heads, ff_dim, dropout) for _ in range(n_layers)])
        self.norm   = nn.LayerNorm(embed_dim)
        self.grad_ckpt = False
        self.out_dim   = embed_dim

    def forward(self, x):
        z = self.cnn(x).transpose(1, 2)
        c = self.pos(self.proj(z))
        for layer in self.layers:
            if self.grad_ckpt and self.training:
                c = torch.utils.checkpoint.checkpoint(layer, c, use_reentrant=False)
            else:
                c = layer(c)
        return self.norm(c).mean(dim=1)


# ── MERL's 1-D ResNet18 (github.com/cheliu-computation/MERL-ICML2024, utils/resnet1d.py, MIT licence) ──
class BasicBlock1d(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(nn.Conv1d(in_channels, out_channels, 1, stride, bias=False), nn.BatchNorm1d(out_channels))

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        return torch.relu(self.bn2(self.conv2(out)) + self.shortcut(x))


class ResNet1d(nn.Module):
    def __init__(self, num_blocks=(2, 2, 2, 2)):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv1d(12, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.layer1 = self._make_layer(64, num_blocks[0], 1)
        self.layer2 = self._make_layer(128, num_blocks[1], 2)
        self.layer3 = self._make_layer(256, num_blocks[2], 2)
        self.layer4 = self._make_layer(512, num_blocks[3], 2)

    def _make_layer(self, out_channels, n, stride):
        layers = []
        for st in [stride] + [1] * (n - 1):
            layers.append(BasicBlock1d(self.in_channels, out_channels, st))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        return self.layer4(self.layer3(self.layer2(self.layer1(x))))


class ResNetECGEncoder(nn.Module):
    """MERL ResNet18 backbone + global average pooling -> [B, 512]."""
    def __init__(self):
        super().__init__()
        self.net = ResNet1d()
        self.out_dim = 512
        self.grad_ckpt = False

    def forward(self, x):
        return self.net(x).mean(dim=-1)


def build_ecg_encoder():
    return ResNetECGEncoder() if ECG_ARCH == 'resnet18' else ECGEncoder()


_enc = build_ecg_encoder().to(DEVICE)
print(f'ECG_ARCH={ECG_ARCH} | output:', _enc(torch.randn(2, 12, 2500, device=DEVICE)).shape)
print(f'ECG encoder params: {sum(p.numel() for p in _enc.parameters()):,}')
del _enc

## 5. Text Encoder (Bio_ClinicalBERT or MedCPT)

In [ ]:
from transformers import AutoTokenizer, AutoModel


class ClinicalTextEncoder(nn.Module):
    """Bio_ClinicalBERT; returns the [CLS] token embedding [B, 768]."""
    def __init__(self, model_name=TEXT_MODEL_NAME, freeze=False):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask, **kwargs):   # **kwargs absorbs token_type_ids
        return self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]


tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
# Pre-tokenise every text once (no padding here; batches are padded dynamically)
TOKENS = tokenizer(texts, padding=False, truncation=True, max_length=TEXT_MAX_LEN)['input_ids']
print('Tokens per text: mean', round(float(np.mean([len(t) for t in TOKENS])), 1), '| max', max(len(t) for t in TOKENS))

## 6. Multimodal CLIP Model

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, proj_dim=PROJ_DIM, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, in_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(in_dim, proj_dim))

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class ECGTextCLIP(nn.Module):
    def __init__(self, ecg_encoder, text_encoder, proj_dim=PROJ_DIM, init_temp=INIT_TEMP, uma=False):
        super().__init__()
        self.ecg_encoder, self.text_encoder = ecg_encoder, text_encoder
        self.ecg_proj  = ProjectionHead(ecg_encoder.out_dim, proj_dim)
        if uma:   # MERL uni-modal alignment: two bias-free maps of the pooled ECG feature + dropout 0.1
            self.uma1 = nn.Linear(ecg_encoder.out_dim, 256, bias=False)
            self.uma2 = nn.Linear(ecg_encoder.out_dim, 256, bias=False)
            self.uma_drop = nn.Dropout(0.1)
        self.text_proj = ProjectionHead(text_encoder.embed_dim, proj_dim)
        self.log_temp  = nn.Parameter(torch.tensor(math.log(init_temp), dtype=torch.float32))

    @property
    def temperature(self):
        return self.log_temp.exp().clamp(min=0.01, max=1.0)

    def encode_ecg(self, ecg):
        return self.ecg_proj(self.ecg_encoder(ecg))

    def encode_text(self, input_ids, attention_mask, **kwargs):
        return self.text_proj(self.text_encoder(input_ids, attention_mask))

    def forward(self, ecg, input_ids, attention_mask, **kwargs):
        return self.encode_ecg(ecg), self.encode_text(input_ids, attention_mask), self.temperature

    def uma_views(self, feat):
        return (F.normalize(self.uma_drop(self.uma1(feat)), dim=-1),
                F.normalize(self.uma_drop(self.uma2(feat)), dim=-1))

## 7. Contrastive Loss (CLIP / multi-positive)

In [ ]:
def clip_loss(ecg_emb, text_emb, temperature, text_ids=None):
    """Symmetric InfoNCE. With text_ids, every pair of samples that share the exact same text is a
    positive (soft target spread over them) instead of a false negative. With many identical reports
    (e.g. 'sinus rhythm. normal ecg.') standard CLIP would push identical-report ECGs apart."""
    logits = (ecg_emb @ text_emb.T) / temperature
    if text_ids is None:
        labels = torch.arange(ecg_emb.size(0), device=ecg_emb.device)
        return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
    pos = (text_ids[:, None] == text_ids[None, :]).float()
    tgt = pos / pos.sum(1, keepdim=True)
    l_e2t = -(tgt * F.log_softmax(logits, dim=1)).sum(1).mean()
    l_t2e = -(tgt * F.log_softmax(logits.T, dim=1)).sum(1).mean()
    return (l_e2t + l_t2e) / 2

## 8. Dataset, Splits & Loaders

In [ ]:
from torch.utils.data import Dataset, DataLoader


class ECGTextDataset(Dataset):
    """Training: random 5-s crop anywhere in the 10-s record (augmentation).
    Evaluation: the full 10 s, embedded later as the average of its two 5-s windows (deterministic)."""
    def __init__(self, idx, train):
        self.idx, self.train = np.asarray(idx), train

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        x = SIG[j]
        if self.train:
            s = random.randint(0, x.shape[1] - SEGMENT_LEN)
            x = x[:, s:s + SEGMENT_LEN]
        return torch.from_numpy(np.ascontiguousarray(x)).float(), j


def collate(batch):
    ecg = torch.stack([b[0] for b in batch])
    j = np.array([b[1] for b in batch])
    L = max(len(TOKENS[k]) for k in j)
    ids = torch.zeros(len(j), L, dtype=torch.long)
    att = torch.zeros(len(j), L, dtype=torch.long)
    for r, k in enumerate(j):
        ids[r, :len(TOKENS[k])] = torch.tensor(TOKENS[k]); att[r, :len(TOKENS[k])] = 1
    return {'ecg': ecg, 'input_ids': ids, 'attention_mask': att,
            'text_ids': torch.from_numpy(TEXT_IDS[j]), 'row': torch.from_numpy(j)}


fold = ptb_df['strat_fold'].values
train_idx = np.where(fold <= 8)[0]      # official PTB-XL split
val_idx   = np.where(fold == 9)[0]
test_idx  = np.where(fold == 10)[0]
print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

train_loader = DataLoader(ECGTextDataset(train_idx, train=True), batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=0, collate_fn=collate)
batch = next(iter(train_loader))
print('ecg', tuple(batch['ecg'].shape), '| input_ids', tuple(batch['input_ids'].shape),
      '| identical-text pairs in batch:', int((batch['text_ids'][:, None] == batch['text_ids'][None]).sum() - BATCH_SIZE))

## 9. Training

In [ ]:
from sklearn.metrics import roc_auc_score

AMP = dict(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16 and DEVICE == 'cuda')

# Zero-shot prompts. 'scp': the SCP descriptions of each superclass' diagnostic statements.
# 'handwritten': the v1 templates (kept for comparison).
SCP_PROMPTS = {s: [d + '.' for d in diag[diag['diagnostic_class'] == s]['description']] for s in SUPERS}
# MERL's released CKEPE prompts: one comma-separated list of subtypes per class
CKEPE_PROMPTS = None
if os.path.exists(CKEPE_JSON):
    _ck = json.load(open(CKEPE_JSON, encoding='utf-8'))
    CKEPE_PROMPTS = {s: [_ck[s].lower()] for s in SUPERS}
HANDWRITTEN_PROMPTS = {
    'NORM': ['Normal sinus rhythm. No significant abnormalities.', 'Normal ECG. Regular heart rate and rhythm.'],
    'MI':   ['Myocardial infarction. ST elevation with Q waves.', 'Acute myocardial infarction. Ischemic changes on ECG.'],
    'STTC': ['ST-T wave changes. Non-specific repolarization abnormality.', 'T-wave inversion. ST depression.'],
    'CD':   ['Conduction disturbance. Bundle branch block.', 'Left bundle branch block. Prolonged QRS duration.'],
    'HYP':  ['Ventricular hypertrophy. Left ventricular hypertrophy with high voltage.', 'Right ventricular hypertrophy. Axis deviation.'],
}


@torch.no_grad()
def embed_ecgs(model, idx, bs=128):
    """Returns (encoder features [N, 768], projected embeddings [N, 512]); average of both 5-s windows."""
    model.eval()
    feats, projs = [], []
    loader = DataLoader(ECGTextDataset(idx, train=False), batch_size=bs, shuffle=False, num_workers=0)
    for x, _ in loader:
        x = x.to(DEVICE)
        xs = torch.cat([x[..., :SEGMENT_LEN], x[..., SEGMENT_LEN:]], 0)
        with torch.autocast(**AMP):
            f = model.ecg_encoder(xs).float()
        p = model.ecg_proj(f)
        n = x.size(0)
        feats.append(((f[:n] + f[n:]) / 2).cpu())
        projs.append(F.normalize(p[:n] + p[n:], dim=-1).cpu())
    return torch.cat(feats).numpy(), torch.cat(projs)


@torch.no_grad()
def embed_texts(model, text_list, bs=256):
    model.eval()
    out = []
    for i in range(0, len(text_list), bs):
        tok = tokenizer(text_list[i:i + bs], return_tensors='pt', padding=True, truncation=True,
                        max_length=TEXT_MAX_LEN).to(DEVICE)
        with torch.autocast(**AMP):
            out.append(model.encode_text(**tok).float().cpu())
    return F.normalize(torch.cat(out), dim=-1)


def zero_shot_scores(model, ecg_proj, prompts, agg='max'):
    """[N, 5] similarity of every ECG to each superclass. agg='max': best-matching prompt of the class."""
    cols = []
    for s in SUPERS:
        te = embed_texts(model, prompts[s])
        cols.append((ecg_proj @ te.T).max(1).values if agg == 'max' and len(te) > 1 else ecg_proj @ F.normalize(te.mean(0), dim=-1))
    return torch.stack(cols, 1).numpy()


def macro_auc(Ytrue, S):
    per = [roc_auc_score(Ytrue[:, k], S[:, k]) for k in range(len(SUPERS))]
    return float(np.mean(per)), per


def save_checkpoint(path, model, optimizer, epoch, history):
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch, 'history': history}, path)


def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    if optimizer is not None:
        optimizer.load_state_dict(ckpt['optimizer'])
    return ckpt.get('epoch', 0), ckpt.get('history', {})

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────
ecg_encoder  = build_ecg_encoder()
text_encoder = ClinicalTextEncoder(freeze=FREEZE_TEXT)
model = ECGTextCLIP(ecg_encoder, text_encoder, uma=USE_UMA).to(DEVICE)
if GRAD_CKPT and not FREEZE_TEXT:
    model.ecg_encoder.grad_ckpt = True
    model.text_encoder.bert.gradient_checkpointing_enable()

def map_cmsc_checkpoint(ckpt):
    """Pretrain.ipynb (CMSC) names its modules differently (feature_extractor, transformer_layers, final_norm, ...).
    Map them onto this notebook's ECGEncoder names. v1 used strict=False here, which silently loaded NOTHING."""
    sd = ckpt.get('model_state_dict', ckpt)
    rules = [('feature_extractor.input_proj.', 'cnn.proj.'), ('feature_extractor.input_norm.', 'cnn.norm.'),
             ('feature_extractor.blocks.', 'cnn.blocks.'), ('project_to_transformer.', 'proj.'),
             ('pos_conv.in_proj.', 'pos.in_.'), ('pos_conv.out_proj.', 'pos.out_.'), ('pos_conv.conv.', 'pos.conv.'),
             ('transformer_layers.', 'layers.'), ('final_norm.', 'norm.')]
    out = {}
    for k, v in sd.items():
        if not k.startswith('encoder.'):
            continue                      # drop the CMSC projection head
        k = k[len('encoder.'):]
        for a, b in rules:
            if k.startswith(a):
                k = b + k[len(a):]
                break
        out[k] = v
    return out
# ── end map


# Optional: warm-start the ECG encoder from a Pretrain.ipynb (CMSC) checkpoint, e.g. 'ecgfm_epoch_10.pt'
ECG_PRETRAINED = None
if ECG_PRETRAINED and os.path.exists(ECG_PRETRAINED):
    ckpt = torch.load(ECG_PRETRAINED, map_location='cpu')
    print(model.ecg_encoder.load_state_dict(map_cmsc_checkpoint(ckpt), strict=True))   # fails loudly on mismatch

param_groups = [
    {'params': model.ecg_encoder.parameters(),  'base_lr': LR_ECG},
    {'params': [p for p in model.text_encoder.parameters() if p.requires_grad], 'base_lr': LR_TEXT},
    {'params': list(model.ecg_proj.parameters()) + list(model.text_proj.parameters())
               + ([model.uma1.weight, model.uma2.weight] if USE_UMA else []), 'base_lr': LR_PROJ},
    {'params': [model.log_temp], 'base_lr': LR_TEMP, 'weight_decay': 0.0},
]
param_groups = [g for g in param_groups if len(list(g['params'])) > 0]
for g in param_groups:
    g['lr'] = g['base_lr']
optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(train_loader)
total_steps  = NUM_EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch


def set_lr(step):
    """Linear warm-up, then cosine decay to MIN_LR_FRAC of each group's base LR."""
    if step < warmup_steps:
        scale = step / max(1, warmup_steps)
    else:
        scale = 0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / max(1, total_steps - warmup_steps)))
    for g in optimizer.param_groups:
        g['lr'] = g['base_lr'] * max(scale, MIN_LR_FRAC)


history = {'train_loss': [], 'val_zs_auc': [], 'temperature': []}
best_val_auc, global_step = -1.0, 0
RESUME_PATH = None
start_epoch = 0
if RESUME_PATH and os.path.exists(RESUME_PATH):
    start_epoch, history = load_checkpoint(RESUME_PATH, model, optimizer)
    global_step = start_epoch * steps_per_epoch

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    losses, t0 = [], time.time()
    for batch in train_loader:
        set_lr(global_step)
        ecg  = batch['ecg'].to(DEVICE, non_blocking=True)
        ids  = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        tids = batch['text_ids'].to(DEVICE) if MULTI_POSITIVE else None

        with torch.autocast(**AMP):
            feat     = model.ecg_encoder(ecg)
            ecg_emb  = model.ecg_proj(feat)
            text_emb = model.encode_text(ids, mask)
        loss = clip_loss(ecg_emb.float(), text_emb.float(), model.temperature, tids)
        if USE_UMA:   # MERL: fixed temperature 0.07 for the uni-modal term
            u1, u2 = model.uma_views(feat.float())
            loss = loss + clip_loss(u1, u2, torch.tensor(0.07, device=DEVICE))

        optimizer.zero_grad(set_to_none=True)
        if not torch.isfinite(loss):
            print(f'  [step {global_step}] non-finite loss, skipping batch'); global_step += 1; continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        losses.append(loss.item()); global_step += 1
        if global_step % 50 == 0:
            print(f'  epoch {epoch+1} step {global_step}/{total_steps} loss {np.mean(losses[-50:]):.4f} '
                  f'temp {model.temperature.item():.4f} lr_ecg {optimizer.param_groups[0]["lr"]:.2e}')

    # ── validation: zero-shot macro AUROC (model selection) ────────────────
    _, val_proj = embed_ecgs(model, val_idx)
    val_auc, _ = macro_auc(Y[val_idx], zero_shot_scores(model, val_proj, SCP_PROMPTS))
    history['train_loss'].append(float(np.mean(losses)))
    history['val_zs_auc'].append(val_auc)
    history['temperature'].append(model.temperature.item())
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | train_loss {np.mean(losses):.4f} | val zero-shot AUROC {val_auc:.4f} | '
          f'temp {model.temperature.item():.4f} | {(time.time()-t0)/60:.1f} min')

    if val_auc > best_val_auc:   # best.pt = weights only (~0.8 GB); last.pt also has the optimizer (~2.3 GB) for resuming
        best_val_auc = val_auc
        torch.save({'model': model.state_dict(), 'epoch': epoch + 1, 'history': history}, os.path.join(SAVE_DIR, 'best.pt'))
    save_checkpoint(os.path.join(SAVE_DIR, 'last.pt'), model, optimizer, epoch + 1, history)

print(f'Training complete. Best val zero-shot AUROC: {best_val_auc:.4f}')

## 10. Training Curves

In [ ]:
import matplotlib.pyplot as plt

ep = range(1, len(history['train_loss']) + 1)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(ep, history['train_loss']); ax[0].set_title('Train contrastive loss'); ax[0].set_xlabel('Epoch')
ax[1].plot(ep, history['val_zs_auc']); ax[1].set_title('Val zero-shot macro AUROC'); ax[1].set_xlabel('Epoch')
ax[2].plot(ep, history['temperature']); ax[2].set_title('Learned temperature τ'); ax[2].set_xlabel('Epoch')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'curves.png'), dpi=150); plt.show()

## 11. Evaluation — Zero-Shot Classification
No labelled ECG is used: each ECG is scored by its cosine similarity to the text prompts of each superclass.
Reported as **macro AUROC** (standard PTB-XL multi-label metric) and, for the records with exactly one superclass,
accuracy / F1 with **correctly named** rows.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

load_checkpoint(os.path.join(SAVE_DIR, 'best.pt'), model)
test_feat, test_proj = embed_ecgs(model, test_idx)
Y_test = Y[test_idx]

prompt_sets = [('SCP-description prompts', SCP_PROMPTS), ('Hand-written prompts (v1)', HANDWRITTEN_PROMPTS)]
if CKEPE_PROMPTS is not None:
    prompt_sets.append(('MERL CKEPE prompts', CKEPE_PROMPTS))
for name, prompts in prompt_sets:
    S = zero_shot_scores(model, test_proj, prompts)
    auc, per = macro_auc(Y_test, S)
    print(f'{name}: macro AUROC {auc:.4f} | ' + ', '.join(f'{s} {a:.3f}' for s, a in zip(SUPERS, per)))

S = zero_shot_scores(model, test_proj, SCP_PROMPTS)
single = Y_test.sum(1) == 1
y_true, y_pred = Y_test[single].argmax(1), S[single].argmax(1)
print(f'\nSingle-label test records: {single.sum()} | accuracy {accuracy_score(y_true, y_pred):.4f}')
print(classification_report(y_true, y_pred, labels=list(range(len(SUPERS))), target_names=SUPERS, zero_division=0))

## 12. Linear Probe (1% / 10% / 100% labels) vs. Random-Init Baseline
Freeze the ECG encoder, fit one logistic regression per superclass on the 768-d features, report test macro AUROC.
The **random-init** row is the same architecture with untrained weights: pre-training only helps if it beats this row.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


def linear_probe(Xtr, Ytr, Xva, Yva, Xte, Yte, frac, seed=0):
    rng = np.random.RandomState(seed)
    sel = rng.choice(len(Xtr), max(int(len(Xtr) * frac), 50), replace=False)
    sc = StandardScaler().fit(Xtr[sel])
    a, b, c = sc.transform(Xtr[sel]), sc.transform(Xva), sc.transform(Xte)
    best = None
    for C in [0.001, 0.01, 0.1, 1.0]:            # C chosen on the validation fold
        va, te = [], []
        for k in range(len(SUPERS)):
            clf = LogisticRegression(C=C, max_iter=2000).fit(a, Ytr[sel, k])
            va.append(roc_auc_score(Yva[:, k], clf.decision_function(b)))
            te.append(roc_auc_score(Yte[:, k], clf.decision_function(c)))
        if best is None or np.mean(va) > best[0]:
            best = (np.mean(va), np.mean(te))
    return best[1]


rows = {}
rand_model = ECGTextCLIP(build_ecg_encoder(), model.text_encoder).to(DEVICE)   # untrained ECG encoder
for name, m in [('CLIP-pretrained', model), ('Random init', rand_model)]:
    Xtr, _ = embed_ecgs(m, train_idx)
    Xva, _ = embed_ecgs(m, val_idx)
    Xte, _ = embed_ecgs(m, test_idx)
    rows[name] = {f'{int(f*100)}%': round(linear_probe(Xtr, Y[train_idx], Xva, Y[val_idx], Xte, Y[test_idx], f), 4)
                  for f in [0.01, 0.1, 1.0]}
del rand_model
print(pd.DataFrame(rows).T.rename_axis('Linear probe test macro AUROC'))

## 13. Embedding Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

single = Y_test.sum(1) == 1
emb, lab = test_proj.numpy()[single], Y_test[single].argmax(1)
if len(emb) > 2000:
    keep = np.random.RandomState(0).choice(len(emb), 2000, replace=False)
    emb, lab = emb[keep], lab[keep]
emb2d = TSNE(n_components=2, perplexity=40, max_iter=1000, random_state=42).fit_transform(emb)

fig, ax = plt.subplots(figsize=(9, 7))
for k, s in enumerate(SUPERS):
    ax.scatter(emb2d[lab == k, 0], emb2d[lab == k, 1], s=8, alpha=0.6, label=s)
ax.legend(markerscale=3); ax.set_title('t-SNE of test ECG embeddings (single-superclass records)')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'tsne.png'), dpi=150); plt.show()

## 14. Text-to-ECG Retrieval Demo

In [ ]:
@torch.no_grad()
def retrieve(query, k=10):
    q = embed_texts(model, [query])
    sims = (q @ test_proj.T).squeeze(0)
    top = sims.topk(k).indices.numpy()
    print(f'Query: "{query}"')
    for r, t in enumerate(top, 1):
        row = test_idx[t]
        labels = [s for s, v in zip(SUPERS, Y[row]) if v] or ['-']
        print(f'  #{r:2d} sim={sims[t]:.3f} labels={",".join(labels):12s} report: {texts[row][:90]}')
    return top


for q in ['anterior myocardial infarction.', 'complete left bundle branch block.', 'left ventricular hypertrophy.']:
    retrieve(q, k=5); print()

## 15. Benchmark against MERL (optional, script)
`src/merl_bench.py` evaluates any checkpoint of this notebook (and MERL's released checkpoint) on MERL's own splits —
PTB-XL super / sub / form / rhythm, CSN and CPSC2018 — with zero-shot (SCP / CKEPE prompts) and a 1 / 10 / 100 % linear probe:

```bash
python src/merl_bench.py cache
python src/merl_bench.py eval --model ours:<run_dir_with_best.pt_and_log.txt>
python src/merl_bench.py eval --model merl:merl_ref/ckpt/res18_best_ckpt.pth
python src/merl_bench.py table
```